## match ID 가져오기 <추가데이터 확보하기 - 2024 ~ 2025년도 매치데이터 >
* 2023년도의 매치 데이터는 라이엇 서버에서 저장하고 있지 않아서 가져올 수 없음 

In [8]:
import pandas as pd
merged_df = pd.read_csv("merged_df.csv")

In [9]:
import requests
import pandas as pd
import time
import dotenv
import os

from datetime import datetime

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

# =========================================================
# match ids 수집 함수
# - 2024 ~ 2025 시즌 전용
# - 솔로랭크(queue=420)만 수집
# =========================================================
def get_match_ids(
    puuid,
    start_time,
    end_time,
    total_count=300,
    queue=420,
    max_retry=3
):

    # =====================================================
    # puuid 검증
    # =====================================================
    if pd.isna(puuid) or puuid is None or puuid == "":

        print("[INVALID PUUID]")

        return []

    headers = {
        "X-Riot-Token": API_KEY
    }

    all_match_ids = []

    # =====================================================
    # pagination
    # =====================================================
    for start in range(0, total_count, 100):

        count = min(100, total_count - start)

        url = (
            f"https://asia.api.riotgames.com"
            f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
            f"?start={start}"
            f"&count={count}"
            f"&queue={queue}"
            f"&startTime={start_time}"
            f"&endTime={end_time}"
        )

        retry = 0

        while retry < max_retry:

            try:

                r = requests.get(
                    url,
                    headers=headers,
                    timeout=10
                )

                # =========================================
                # 정상 응답
                # =========================================
                if r.status_code == 200:

                    data = r.json()

                    if isinstance(data, list):

                        all_match_ids.extend(data)

                        print(
                            f"[SUCCESS] "
                            f"start={start} "
                            f"/ {len(data)}개"
                        )

                        break

                    else:

                        print("[INVALID RESPONSE]")

                        break

                # =========================================
                # 인증 실패
                # =========================================
                elif r.status_code == 401:

                    print("[401] API KEY 만료")

                    return []

                # =========================================
                # 존재하지 않음
                # =========================================
                elif r.status_code == 404:

                    print("[404] NOT FOUND")

                    break

                # =========================================
                # Rate Limit
                # =========================================
                elif r.status_code == 429:

                    retry_after = r.headers.get(
                        "Retry-After"
                    )

                    wait_time = (
                        int(retry_after)
                        if retry_after
                        else 120
                    )

                    print(
                        f"[429] RATE LIMIT "
                        f"{wait_time}초 대기"
                    )

                    time.sleep(wait_time)

                    retry += 1

                # =========================================
                # 서버 오류
                # =========================================
                elif r.status_code in [500, 502, 503, 504]:

                    print(
                        f"[SERVER ERROR] "
                        f"{r.status_code}"
                    )

                    retry += 1

                    time.sleep(10)

                # =========================================
                # 잘못된 요청
                # =========================================
                elif r.status_code == 400:

                    print("[400] BAD REQUEST")

                    break

                # =========================================
                # 기타 오류
                # =========================================
                else:

                    print(
                        f"[ERROR] STATUS: "
                        f"{r.status_code}"
                    )

                    break

            # =============================================
            # Timeout
            # =============================================
            except requests.exceptions.Timeout:

                print("[TIMEOUT] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # Connection Error
            # =============================================
            except requests.exceptions.ConnectionError:

                print("[CONNECTION ERROR] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # 기타 예외
            # =============================================
            except Exception as e:

                print(f"[EXCEPTION] {e}")

                break

        # =================================================
        # Riot API 보호
        # =================================================
        time.sleep(1.2)

    # =====================================================
    # 중복 제거 (순서 유지)
    # =====================================================
    all_match_ids = list(
        dict.fromkeys(all_match_ids)
    )

    return all_match_ids


# =========================================================
# 시즌 timestamp
# =========================================================

SEASONS = {

    2023: {
        "start": int(
            datetime(2023, 1, 1).timestamp()
        ),
        "end": int(
            datetime(2023, 12, 31, 23, 59, 59).timestamp()
        )
    },

    2024: {
        "start": int(
            datetime(2024, 1, 1).timestamp()
        ),
        "end": int(
            datetime(2024, 12, 31, 23, 59, 59).timestamp()
        )
    },

    2025: {
        "start": int(
            datetime(2025, 1, 1).timestamp()
        ),
        "end": int(
            datetime(2025, 12, 31, 23, 59, 59).timestamp()
        )
    }

}


# =========================================================
# 선수별 시즌별 match id 수집
# merged_df 안에:
# playername / puuid
# 있다고 가정
# =========================================================

all_rows = []

for idx, row in merged_df.iterrows():

    player = row["playername"]
    puuid = row["puuid"]

    print("\n=================================================")
    print(f"PLAYER: {player}")
    print("=================================================")

    for year, season_info in SEASONS.items():

        print(f"\n[{year}] 수집 시작")

        match_ids = get_match_ids(

            puuid=puuid,

            start_time=season_info["start"],

            end_time=season_info["end"],

            total_count=300
        )

        print(
            f"[{year}] "
            f"{len(match_ids)}개 수집 완료"
        )

        for match_id in match_ids:

            all_rows.append({

                "playername": player,

                "puuid": puuid,

                "game_year": year,

                "match_id": match_id

            })


# =========================================================
# DataFrame 변환
# =========================================================

match_ids_df = pd.DataFrame(all_rows)

print("\n===================================")
print("최종 결과")
print("===================================")

print(match_ids_df.head())

print(f"\nTOTAL ROWS: {len(match_ids_df)}")

print(
    f"UNIQUE MATCHES: "
    f"{match_ids_df['match_id'].nunique()}"
)

print(
    f"UNIQUE PLAYERS: "
    f"{match_ids_df['playername'].nunique()}"
)


# =========================================================
# 저장
# =========================================================

match_ids_df.to_csv(

    "pro_match_ids_2024_2025.csv",

    index=False,

    encoding="utf-8-sig"
)

print("\nCSV 저장 완료")


PLAYER: Canyon

[2023] 수집 시작
[SUCCESS] start=0 / 0개
[SUCCESS] start=100 / 0개
[SUCCESS] start=200 / 0개
[2023] 0개 수집 완료

[2024] 수집 시작
[SUCCESS] start=0 / 0개


KeyboardInterrupt: 

In [8]:
# =========================================================
# DataFrame 변환
# =========================================================

match_df = pd.read_csv("pro_match_ids_2024_2025.csv")

print("=" * 50)
print("최종 match 수:", len(match_df))
print("=" * 50)

최종 match 수: 13568


In [9]:
### 년도별 match 수
match_counts = match_df['game_year'].value_counts()
match_counts 

game_year
2025    10178
2024     3390
Name: count, dtype: int64

In [10]:
### 선수별 match 수
player_match_counts = match_df['playername'].value_counts()
player_match_counts

playername
Croco        1097
Cuzz          881
FATE          860
GIDEON        820
Ucal          745
HamBak        633
YoungJae      628
Fisher        573
Juhan         569
kyeahoo       549
Sylvie        537
Willer        516
SeTab         360
Quad          346
Clid          345
UmTi          340
ShowMaker     338
Ellim         308
Lucid         300
Scout         300
Oner          294
VicLa         246
Pyosik        231
Clozer        213
Loki          207
Peanut        191
Chovy         174
Karis         163
Bdd           162
Raptor        147
Kanavi        121
Sponge        106
Zeka           99
Canyon         64
Grizzly        53
Guwon          47
Pullbae         4
Roamer          1
Name: count, dtype: int64

* 각자 데이터 수가 다른 이유는 계정이 여러개라 그런거임 

In [11]:
print("중복 제거 전:", match_df.shape)

print(
    "중복 수:",
    match_df['match_id'].duplicated().sum()
)

match_df = match_df.drop_duplicates(
    subset=['match_id']
)

print("중복 제거 후:", match_df.shape)

중복 제거 전: (13568, 4)
중복 수: 926
중복 제거 후: (12642, 4)


In [12]:
match_df.groupby('playername')['match_id'].nunique()

playername
Bdd           161
Canyon         64
Chovy         165
Clid          341
Clozer        173
Croco        1097
Cuzz          862
Ellim         297
FATE          854
Fisher        493
GIDEON        789
Grizzly        48
Guwon          26
HamBak        597
Juhan         556
Kanavi        108
Karis         147
Loki          186
Lucid         193
Oner          291
Peanut        191
Pullbae         4
Pyosik        214
Quad          343
Raptor        120
Roamer          1
Scout         279
SeTab         297
ShowMaker     336
Sponge         73
Sylvie        481
Ucal          649
UmTi          338
VicLa         143
Willer        484
YoungJae      627
Zeka           99
kyeahoo       515
Name: match_id, dtype: int64

In [13]:
len(match_df['match_id'].unique()) 

12642

In [ ]:
match_df.to_csv("2025년이전_매치ids.csv", index=False, encoding="utf-8-sig")

### 추가 데이터 분석임 
* 이제 선수 단위로 분석 시작함 
* match ID를 기준으로 API req 보내기 

### match Detail 가져오기 

In [6]:
# 파일 불러오기
import pandas as pd

match_df = pd.read_csv("2025년이전_매치ids.csv")

import os
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

In [10]:
import requests
import time

def get_match_detail(
    match_id,
    max_retry=3
):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}"
    )

    headers = {
        "X-Riot-Token": API_KEY
    }

    retry = 0

    while retry < max_retry:

        try:

            r = requests.get(
                url,
                headers=headers,
                timeout=10
            )

            # =====================================
            # 성공
            # =====================================
            if r.status_code == 200:

                return r.json()

            # =====================================
            # Rate Limit
            # =====================================
            elif r.status_code == 429:

                retry_after = r.headers.get(
                    "Retry-After"
                )

                wait_time = (
                    int(retry_after)
                    if retry_after
                    else 120
                )

                print(
                    f"[429] {wait_time}초 대기"
                )

                time.sleep(wait_time)

                retry += 1

            # =====================================
            # 인증 오류
            # =====================================
            elif r.status_code == 401:

                print("[401] API KEY 오류")

                return None

            # =====================================
            # 존재하지 않음
            # =====================================
            elif r.status_code == 404:

                print(f"[404] {match_id}")

                return None

            # =====================================
            # 서버 오류
            # =====================================
            elif r.status_code in [500, 502, 503, 504]:

                print(
                    f"[SERVER ERROR] "
                    f"{r.status_code}"
                )

                retry += 1

                time.sleep(10)

            else:

                print(
                    f"[ERROR] {r.status_code}"
                )

                return None

        except Exception as e:

            print(f"[EXCEPTION] {e}")

            retry += 1

            time.sleep(5)

    print(f"[FAILED] {match_id}")

    return None

<!-- match_data['info']['participants'] -->

## match data API로 가져오기 
* "그 10명 중에서 내 프로 선수 puuid와 일치하는 participant만 추출"

## match data 파싱 
match_data['info']['participants']
-> 대충 이런식으로 10명의 정보가 다 들어있는데 
-> 여기서 선수들의 퍼포먼스만 볼 예정임 
-> 프로선수 participant만 추출 예정 

* 수집할 컬럼 이름 

| Oracle            | Riot match-v5                   |
| ----------------- | ------------------------------- |
| kills             | kills                           |
| deaths            | deaths                          |
| assists           | assists                         |
| visionscore       | visionScore                     |
| earnedgold        | goldEarned                      |
| cspm              | totalMinionsKilled / gameLength |
| damagetochampions | totalDamageDealtToChampions     |
| damagetaken       | totalDamageTaken                |
| wardsplaced       | wardsPlaced                     |
| wardskilled       | wardsKilled                     |


In [11]:
# # match_ids 랑 pro_match_ids_2024_2025.csv 합치기 만약 중복된 match_id가 있으면 match_ids.csv는 버리고 pro_match_ids_2024_2025.csv로 대체하기

# a = pd.read_csv("match_ids.csv")
# b = pd.read_csv("pro_match_ids_2024_2025.csv")

# match_ids = pd.concat([a,b], ignore_index=True)
# match_ids.drop_duplicates(subset=['match_id'], keep='last', inplace=True)
# match_ids.to_csv("merged_match_ids.csv", index=False, encoding="utf-8-sig")
# print("병합 및 중복 제거 완료")
# print(f"최종 match 수: {len(match_ids)}")
import pandas as pd

match_df = pd.read_csv("2025년이전_매치ids.csv")

In [12]:
from datetime import datetime


def extract_player_data(match_data, puuid):

    participants = match_data['info']['participants']

    valid_positions = {
        'TOP',
        'JUNGLE',
        'MIDDLE',
        'BOTTOM',
        'UTILITY'
    }

    # ==================================================
    # 게임 메타 정보
    # ==================================================

    info = match_data['info']

    # 패치 버전
    # 예: 15.10.682.1234 -> 15.10
    full_version = info.get('gameVersion', '')
    patch = '.'.join(full_version.split('.')[:2])

    # 게임 날짜
    # Riot는 ms timestamp 사용
    game_creation = info.get('gameCreation')

    dt = datetime.fromtimestamp(
        game_creation / 1000
    )

    game_date = dt.strftime('%Y-%m-%d')
    game_year = dt.year
    game_month = dt.month

    for p in participants:

        if p['puuid'] == puuid:

            team_pos = p.get('teamPosition')
            indiv_pos = p.get('individualPosition')

            # 정상 포지션만
            if team_pos not in valid_positions:
                return None

            # 포지션 불일치 제거
            if team_pos != indiv_pos:
                return None

            game_duration = (
                info['gameDuration']
            )

            # 리메이크 제거
            if game_duration < 600:
                return None

            # 총 CS
            cs = (
                p['totalMinionsKilled']
                + p['neutralMinionsKilled']
            )

            # ==================================================
            # 반환 데이터
            # ==================================================

            filtered_p = {

                # -------------------
                # 메타 정보
                # -------------------

                'match_id':
                    match_data['metadata']['matchId'],

                'patch':
                    patch,

                'game_date':
                    game_date,

                'game_year':
                    game_year,

                'game_month':
                    game_month,

                # -------------------
                # 기본 정보
                # -------------------

                'puuid':
                    p['puuid'],

                'summonerName':
                    p['summonerName'],

                'riotIdGameName':
                    p.get('riotIdGameName'),

                'championName':
                    p['championName'],

                'teamPosition':
                    p['teamPosition'],

                'win':
                    p['win'],

                'gameDuration':
                    game_duration,

                # -------------------
                # 전투
                # -------------------

                'kills':
                    p['kills'],

                'deaths':
                    p['deaths'],

                'assists':
                    p['assists'],

                'doubleKills':
                    p['doubleKills'],

                'tripleKills':
                    p['tripleKills'],

                'quadraKills':
                    p['quadraKills'],

                'pentaKills':
                    p['pentaKills'],

                # -------------------
                # 데미지
                # -------------------

                'totalDamageDealtToChampions':
                    p['totalDamageDealtToChampions'],

                'totalDamageTaken':
                    p['totalDamageTaken'],

                'damageSelfMitigated':
                    p['damageSelfMitigated'],

                'timeCCingOthers':
                    p['timeCCingOthers'],

                # -------------------
                # 시야
                # -------------------

                'visionScore':
                    p['visionScore'],

                'wardsPlaced':
                    p['wardsPlaced'],

                'wardsKilled':
                    p['wardsKilled'],

                'visionWardsBoughtInGame':
                    p['visionWardsBoughtInGame'],

                # -------------------
                # 성장
                # -------------------

                'goldEarned':
                    p['goldEarned'],

                'champLevel':
                    p['champLevel'],

                'cs':
                    cs,

                # -------------------
                # 오브젝트
                # -------------------

                'damageDealtToObjectives':
                    p['damageDealtToObjectives'],

                'damageDealtToTurrets':
                    p['damageDealtToTurrets'],

                'dragonKills':
                    p['dragonKills'],

                'baronKills':
                    p['baronKills'],

                'turretKills':
                    p['turretKills'],

                # -------------------
                # 퍼블
                # -------------------

                'firstBloodKill':
                    p['firstBloodKill'],

                'firstBloodAssist':
                    p['firstBloodAssist'],
            }

            # ==================================================
            # 파생 변수
            # ==================================================

            minutes = game_duration / 60

            filtered_p['kda'] = (
                (p['kills'] + p['assists'])
                / (p['deaths'] + 1)
            )

            filtered_p['cspm'] = (
                cs / minutes
            )

            filtered_p['dpm'] = (
                p['totalDamageDealtToChampions']
                / minutes
            )

            filtered_p['earned_gpm'] = (
                p['goldEarned']
                / minutes
            )

            filtered_p['vspm'] = (
                p['visionScore']
                / minutes
            )

            filtered_p['wpm'] = (
                p['wardsPlaced']
                / minutes
            )

            filtered_p['wcpm'] = (
                p['wardsKilled']
                / minutes
            )

            filtered_p['damagetakenperminute'] = (
                p['totalDamageTaken']
                / minutes
            )

            filtered_p['damagemitigatedperminute'] = (
                p['damageSelfMitigated']
                / minutes
            )

            return filtered_p

    return None

In [13]:
import os
import pandas as pd
import time

# =====================================================
# 저장 경로
# =====================================================

CHECKPOINT_DIR = "API_checkpoint"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "2025년이전_매치ids_data.csv"
)

# =====================================================
# 기존 데이터 불러오기 (복구용)
# =====================================================

if os.path.exists(SAVE_PATH):

    existing_df = pd.read_csv(SAVE_PATH)

    all_player_data = existing_df.to_dict('records')

    collected_match_ids = set(
        existing_df['match_id'].unique()
    )

    print(
        f"기존 데이터 로드 완료 "
        f"({len(collected_match_ids)}개)"
    )

else:

    all_player_data = []

    collected_match_ids = set()

    print("새로운 수집 시작")

# =====================================================
# match detail 수집
# =====================================================

for idx, row in match_df.iterrows():

    try:

        match_id = row['match_id']
        puuid = row['puuid']
        playername = row['playername']

        # =============================================
        # 이미 수집한 경기 skip
        # =============================================
        if match_id in collected_match_ids:

            print(f"[SKIP] {match_id}")

            continue

        # =============================================
        # match detail 가져오기
        # =============================================
        match_data = get_match_detail(match_id)

        if match_data is None:

            print(f"[NO DATA] {match_id}")

            continue

        # =============================================
        # participant 추출
        # =============================================
        player_data = extract_player_data(
            match_data,
            puuid
        )

        if player_data is None:

            print(
                f"[NO PARTICIPANT] "
                f"{playername}"
            )

            continue

        # =============================================
        # 추가 정보 저장
        # =============================================
        player_data['playername'] = playername
        player_data['puuid'] = puuid

        all_player_data.append(player_data)

        collected_match_ids.add(match_id)

        print(
            f"[{idx}] "
            f"{playername} "
            f"- 저장 완료"
        )

        # =============================================
        # 체크포인트 저장
        # =============================================
        if idx % 50 == 0:

            checkpoint_df = pd.DataFrame(
                all_player_data
            )

            checkpoint_df.to_csv(
                SAVE_PATH,
                index=False,
                encoding='utf-8-sig'
            )

            print(
                f"[CHECKPOINT] "
                f"{len(checkpoint_df)}개 저장"
            )

        # =============================================
        # Rate limit 방지
        # =============================================
        # time.sleep(1.2)

    except Exception as e:

        print(
            f"[ERROR] "
            f"{idx} / {e}"
        )

        continue

# =====================================================
# 최종 저장
# =====================================================

final_df = pd.DataFrame(all_player_data)

final_df.to_csv(
    SAVE_PATH,
    index=False,
    encoding='utf-8-sig'
)

print("=" * 50)
print("최종 저장 완료")
print("총 데이터 수:", len(final_df))
print("=" * 50)

새로운 수집 시작
[0] Canyon - 저장 완료
[CHECKPOINT] 1개 저장
[1] Canyon - 저장 완료
[2] Canyon - 저장 완료
[3] Canyon - 저장 완료
[4] Canyon - 저장 완료
[5] Canyon - 저장 완료
[6] Canyon - 저장 완료
[7] Canyon - 저장 완료
[8] Canyon - 저장 완료
[9] Canyon - 저장 완료
[10] Canyon - 저장 완료
[11] Canyon - 저장 완료
[12] Canyon - 저장 완료
[13] Canyon - 저장 완료
[14] Canyon - 저장 완료
[15] Canyon - 저장 완료
[16] Canyon - 저장 완료
[17] Canyon - 저장 완료
[18] Canyon - 저장 완료
[19] Canyon - 저장 완료
[20] Canyon - 저장 완료
[21] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[23] Canyon - 저장 완료
[24] Canyon - 저장 완료
[25] Canyon - 저장 완료
[26] Canyon - 저장 완료
[27] Canyon - 저장 완료
[28] Canyon - 저장 완료
[29] Canyon - 저장 완료
[30] Canyon - 저장 완료
[31] Canyon - 저장 완료
[32] Canyon - 저장 완료
[33] Canyon - 저장 완료
[34] Canyon - 저장 완료
[35] Canyon - 저장 완료
[36] Canyon - 저장 완료
[37] Canyon - 저장 완료
[38] Canyon - 저장 완료
[39] Canyon - 저장 완료
[40] Canyon - 저장 완료
[41] Canyon - 저장 완료
[42] Canyon - 저장 완료
[43] Canyon - 저장 완료
[44] Canyon - 저장 완료
[45] Canyon - 저장 완료
[46] Canyon - 저장 완료
[47] Canyon - 저장 완료
[48] Canyon - 저장 

* 중간 저장할때마다 곂치는게 존재함으로 중복된 데이터에 대한 처리를 해줘야함 
* 하지만 한 게임에 같은 선수들이 있을 수 있다보니 생기는 문제임 
* 같은 경기(match_id)에 프로 여러 명이 있었던 경우
* match_id 가 곂치는 부분은 "프로 간 같은 게임 존재"하는걸로 해석을 할 수 있기에 이 값은 처리를 안하는게 맞음

* `그래도 완전 동일 row 중복이나 , 같은 게임에 플레이어 이름이 2번 들어간 경우는 오류임으로 제거해줌 `

In [14]:
df = pd.read_csv("API_checkpoint/2025년이전_매치ids_data.csv")
print(df.shape)

(12526, 47)


In [15]:
print("원본 데이터 크기:", df.shape)

# 1. 완전 동일 row 중복 확인
full_duplicates = df.duplicated().sum()

print("완전 동일 row 중복 수:", full_duplicates)

# 2. 같은 선수 + 같은 경기 중복 확인
player_match_duplicates = df.duplicated(
    subset=['playername', 'match_id']
).sum()

print(
    "playername + match_id 중복 수:",
    player_match_duplicates
)

# ============================================
# 중복 제거
# ============================================

df_clean = df.drop_duplicates(
    subset=['playername', 'match_id']
)

print("중복 제거 후 데이터 크기:", df_clean.shape)

# ============================================
# 제거된 row 수 확인
# ============================================

removed_rows = len(df) - len(df_clean)

print("제거된 row 수:", removed_rows)

원본 데이터 크기: (12526, 47)
완전 동일 row 중복 수: 0
playername + match_id 중복 수: 0
중복 제거 후 데이터 크기: (12526, 47)
제거된 row 수: 0


In [24]:
a = pd.read_csv("API_checkpoint/2025년이전_매치ids_data.csv")
b = pd.read_csv("API_checkpoint/선수별_시간별_매치데이터.csv")

riot_matchdf = pd.concat([a, b], ignore_index=True)

riot_matchdf.shape

(35061, 47)

In [25]:
### 중복 제거

riot_matchdf.drop_duplicates(inplace=True)
print("중복 제거 후 크기:", riot_matchdf.shape)

### 결측값 확인 
missing_values = riot_matchdf.isnull().sum()
print("결측값 개수:\n", missing_values)

중복 제거 후 크기: (27485, 47)
결측값 개수:
 match_id                           0
patch                              0
game_date                          0
game_year                          0
game_month                         0
puuid                              0
summonerName                   23751
riotIdGameName                     0
championName                       0
teamPosition                       0
win                                0
gameDuration                       0
kills                              0
deaths                             0
assists                            0
doubleKills                        0
tripleKills                        0
quadraKills                        0
pentaKills                         0
totalDamageDealtToChampions        0
totalDamageTaken                   0
damageSelfMitigated                0
timeCCingOthers                    0
visionScore                        0
wardsPlaced                        0
wardsKilled                        0
visio

In [30]:
# summonerName                   23751
# riotIdGameName                     0
# 요 2개가 결측값이긴한데 어짜피 riotid이름이니까 컬럼을 드랍함

# riot_matchdf.drop(columns=['summonerName', 'riotIdGameName'], inplace=True)
missing_values = riot_matchdf.isnull().sum()
print("결측값 개수:\n", missing_values)

# 저장함
riot_matchdf.to_csv("API_checkpoint/최종_매치데이터.csv", index=False, encoding="utf-8-sig")

결측값 개수:
 match_id                       0
patch                          0
game_date                      0
game_year                      0
game_month                     0
puuid                          0
championName                   0
teamPosition                   0
win                            0
gameDuration                   0
kills                          0
deaths                         0
assists                        0
doubleKills                    0
tripleKills                    0
quadraKills                    0
pentaKills                     0
totalDamageDealtToChampions    0
totalDamageTaken               0
damageSelfMitigated            0
timeCCingOthers                0
visionScore                    0
wardsPlaced                    0
wardsKilled                    0
visionWardsBoughtInGame        0
goldEarned                     0
champLevel                     0
cs                             0
damageDealtToObjectives        0
damageDealtToTurrets           0
d

#### 최종 데이터 분포 확인 



In [ ]:
### 1. 시기별 매치수 


### 2. 패치별 매치수


### 3. 선수별 매치수

